# 03 · Aprendizaje no supervisado — Perfiles de centrales

**Pregunta:** sin usar la etiqueta "fuente", ¿las centrales se agrupan solas según *cómo* generan a lo largo del año? ¿Esos grupos coinciden con la tecnología?

**Técnicas del programa:** K-means (Clase 19-20), DBSCAN (Clase 21), métricas de clustering (Clase 22) y reducción de dimensionalidad con SVD/PCA (Clase 23).

## Representación de cada central
Para cada máquina (con al menos 24 meses de historia) se arma un vector de 15 variables:
- **12 valores de estacionalidad:** generación de cada mes dividida por el promedio de esa máquina (1,0 = mes promedio).
- **Escala:** `log1p` de la generación media (qué tan grande es).
- **Variabilidad:** coeficiente de variación (desvío / media).
- **Intermitencia:** fracción de meses con generación cero.

Todas se estandarizan antes de agrupar, porque K-means usa distancias.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from src import config
from src.preprocessing import cargar_datos

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 160)

In [ ]:
d = cargar_datos()
d["mes_num"] = d["mes"].dt.month

n_meses = d.groupby("maquina")["gwh"].transform("size")
media = d.groupby("maquina")["gwh"].transform("mean")
d = d[(n_meses >= 24) & (media > 0)].copy()
d["relativo"] = d["gwh"] / d.groupby("maquina")["gwh"].transform("mean")

estacional = d.pivot_table(index="maquina", columns="mes_num", values="relativo", aggfunc="mean").dropna()
estacional.columns = [f"mes_{c:02d}" for c in estacional.columns]

resumen = d.groupby("maquina")["gwh"].agg(
    log_media=lambda x: np.log1p(x.mean()),
    coef_var=lambda x: x.std() / x.mean(),
    frac_ceros=lambda x: (x == 0).mean())

X = estacional.join(resumen)
meta = d.groupby("maquina")[["fuente", "region", "provincia"]].first().loc[X.index]
print(f"Centrales en el análisis: {len(X)}  |  variables: {X.shape[1]}")
X.head()

In [ ]:
Z = StandardScaler().fit_transform(X)

## 1. ¿Cuántos grupos? Codo y silhouette
- **Inercia (codo):** suma de distancias al centroide; siempre baja al aumentar k, se busca dónde deja de mejorar mucho.
- **Silhouette:** de −1 a 1; mide qué tan cerca está cada punto de su grupo respecto del grupo vecino. Valores de 0,25-0,5 indican estructura débil a moderada.

In [ ]:
ks = range(2, 9)
inercia, sil = [], []
for k in ks:
    km = KMeans(k, n_init=10, random_state=config.SEMILLA).fit(Z)
    inercia.append(km.inertia_)
    sil.append(silhouette_score(Z, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(list(ks), inercia, "o-"); ax[0].set(title="Método del codo", xlabel="k", ylabel="inercia")
ax[1].plot(list(ks), sil, "o-", color="#e07b39"); ax[1].set(title="Silhouette", xlabel="k", ylabel="silhouette medio")
plt.tight_layout(); plt.show()
pd.DataFrame({"k": list(ks), "inercia": inercia, "silhouette": sil}).set_index("k").round(3)

El silhouette es parecido para k = 3, 4 y 5 (~0,33), y apenas más alto en k = 2 y k = 5. Con **k = 4 y k = 5 aparece un grupo de una sola máquina** (un caso atípico, no un perfil). Con **k = 3** uno de los grupos es chico (4 máquinas), pero resulta coherente (ver más abajo), así que lo elijo.
Es una decisión de criterio, no un óptimo estadístico: k = 2 también sería defendible.

In [ ]:
K = 3
km = KMeans(K, n_init=10, random_state=config.SEMILLA).fit(Z)
meta["cluster"] = km.labels_
print("Tamaño de cada grupo:"); print(meta["cluster"].value_counts().sort_index().to_string())
print(f"\nSilhouette (k={K}): {silhouette_score(Z, km.labels_):.3f}")

## 2. Visualización con PCA / SVD (Clase 23)
Se proyectan las 15 variables a 2 dimensiones. PCA es la descomposición en valores singulares (SVD) de los datos centrados.

In [ ]:
pca = PCA(n_components=2, random_state=config.SEMILLA)
P = pca.fit_transform(Z)
print("Varianza explicada por cada componente:", pca.explained_variance_ratio_.round(3),
      "| total:", pca.explained_variance_ratio_.sum().round(3))

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=meta["cluster"].astype(str), palette="Set2", s=45, ax=ax[0])
ax[0].set(title="Grupos de K-means (sin usar la fuente)", xlabel="Componente 1", ylabel="Componente 2")
sns.scatterplot(x=P[:, 0], y=P[:, 1], hue=meta["fuente"], palette="tab10", s=45, ax=ax[1])
ax[1].set(title="Las mismas centrales coloreadas por su fuente real", xlabel="Componente 1", ylabel="")
plt.tight_layout(); plt.show()

## 3. ¿Qué significa cada grupo?
### 3.1 Perfil estacional de cada grupo

In [ ]:
perfil = X.assign(cluster=meta["cluster"]).groupby("cluster")[list(estacional.columns)].mean()
perfil.columns = range(1, 13)

plt.figure(figsize=(10, 4))
for c, fila in perfil.iterrows():
    plt.plot(fila.index, fila.values, "o-", label=f"Grupo {c} (n={int((meta['cluster'] == c).sum())})")
plt.axhline(1, color="gray", ls="--", lw=1)
plt.xticks(range(1, 13)); plt.xlabel("Mes"); plt.ylabel("Índice estacional (1 = promedio)")
plt.title("Perfil de generación mensual por grupo"); plt.legend(); plt.show()

### 3.2 Composición de cada grupo por fuente y región

In [ ]:
print("Fuente dentro de cada grupo (cantidad de centrales):")
display(pd.crosstab(meta["cluster"], meta["fuente"]))
print("Fuente dentro de cada grupo (% del grupo):")
display((pd.crosstab(meta["cluster"], meta["fuente"], normalize="index") * 100).round(0))
print("Región dentro de cada grupo (cantidad de centrales):")
display(pd.crosstab(meta["cluster"], meta["region"]))

## 4. Comparación con DBSCAN (Clase 21)
DBSCAN no necesita fijar k: agrupa por densidad y marca como **ruido** (−1) a los puntos aislados. Es sensible a `eps`, así que pruebo varios valores.

In [ ]:
filas = []
for eps in [1.5, 2.0, 2.5, 3.0, 3.5]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(Z)
    n_cl = len(set(lab)) - (1 if -1 in lab else 0)
    filas.append({"eps": eps, "grupos": n_cl, "ruido": int((lab == -1).sum()),
                  "silhouette": silhouette_score(Z[lab != -1], lab[lab != -1]) if n_cl > 1 else np.nan})
pd.DataFrame(filas).set_index("eps").round(3)

## 5. Conclusiones
**1. La estructura es real pero moderada.** El silhouette de K-means con k = 3 es ~0,33: hay grupos reconocibles, aunque con bordes difusos. Los dos primeros componentes del PCA explican ~61 % de la varianza.

**2. Los grupos se definen por el *patrón anual*, no por la tecnología:**
- **Grupo 0 (142 centrales) — perfil plano.** Índice estacional entre 0,90 y 1,05, baja variabilidad (coef. de variación ~0,33) y casi sin meses en cero (~2 %). Predomina eólica (47 %) y biogás (26 %).
- **Grupo 1 (80 centrales) — muy estacional.** Genera ~0,5 veces su promedio en junio-julio y ~1,5 veces en diciembre. Es mayormente solar (59 %) e hidro (36 %), coherente con la estacionalidad vista en el EDA.
- **Grupo 2 (4 centrales) — generación concentrada de junio a octubre**, con 42 % de meses en cero. Son las 4 de biomasa del Noroeste (Tucumán y Salta). *Hipótesis (el dataset no lo indica):* es consistente con el calendario de la zafra azucarera.

**3. La fuente no determina el grupo.** 17 centrales hidro y 11 solares caen en el grupo de perfil plano. Con estos datos no puedo explicar por qué (faltan potencia, tipo de aprovechamiento hidroeléctrico, ubicación exacta); es una pregunta abierta, no una conclusión.

**4. DBSCAN confirma dos grandes patrones, pero es frágil.** Con `eps` entre 1,5 y 2,5 encuentra 2 grupos y deja 34-60 centrales como ruido (15-27 %); con `eps` ≥ 3 todo colapsa en un solo grupo. Su silhouette (~0,45) **no es comparable** con el de K-means porque se calcula sin el ruido. Además, con `min_samples = 5` un grupo de 4 máquinas no puede formarse por definición.

### Limitaciones
- Solo se agrupan 226 de las 289 centrales (se exigen 24 meses de historia): **las más nuevas quedan afuera**, así que los perfiles reflejan el parque más antiguo.
- Las 12 variables estacionales pesan más en la distancia que las 3 de resumen (escala, variabilidad, ceros); otra ponderación podría cambiar los grupos.
- Los grupos son descriptivos: no prueban causas.

### Posible extensión
Usar el grupo como variable del modelo supervisado. Habría que ajustar el clustering **solo con datos de entrenamiento**, para no filtrar información del test.